In [15]:
"""
서울의 기분 프로젝트 — 스트레스 해결책 크롤러 (네이버 지도)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SUSI 지수(Seoul Urban Stress Index) 4대 분류별 해결 장소 수집

- 환경 스트레스 → 공원, 숲, 자연 힐링 공간
- 교통 스트레스 → 역세권, 접근성 좋은 장소
- 소음 스트레스 → 조용한 카페, 도서관, 한적한 공간
- 안전 스트레스 → 밝고 사람 많은 안전한 장소

[사용법]
1. ★ 네이버 API 키 (Client ID, Secret) 입력

2. ★ 수집할 스트레스 태그 선택 (STRESS_TAGS)
3. 실행 → 자치구별 해결책 장소 CSV 저장
"""

import requests
import pandas as pd

import time
import os
import json

from datetime import datetime
from typing import List, Dict


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ★ ① 네이버 API 인증 정보 입력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NAVER_CLIENT_ID     = "mVTJL4a31K_B27F7UtVC"

NAVER_CLIENT_SECRET = "WN0KGvYpuu"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ★ ② 수집할 스트레스 해결 태그
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRESS_TAGS = {
    # 환경 스트레스 해결책
    "환경": [
        "공원", "한강공원", "서울숲", "산책로", "숲", "생태공원",
        "정원", "수목원", "식물원", "녹지", "힐링", "도시숲", "산",
        "힐링스팟", "피크닉명소", "산책코스", 
        "한강", "도심속자연", "노을명소", "서울힐링",
    ],
    
    # 교통 스트레스 해결책
    "교통": [
        "역세권 카페", "역 앞 맛집", "지하철역", "환승역 카페",
        "접근성 좋은", "역 5분", "주차 편한", "역근처",
        "역 앞", "역 바로 앞", "지하 연결",
    ],
    
    # 소음 스트레스 해결책
    "소음": [
        "조용한 카페", "도서관", "북카페", "한적한 카페",
        "미술관", "박물관", "전시회", "힐링", "수목원", "식물원", "도심속자연",
        "조용한", "차분한", "혼자 가기 좋은",
        "조용히", "한적한", "여유로운", "산책코스"
    ],
    
    # 안전 스트레스 해결책
    "안전": [
        "밝은 카페", "사람 많은", "번화가", "대형 쇼핑몰",
        "백화점", "복합문화공간", "안전한", "CCTV 많은",
        "관리 잘된", "경비", "24시간",
        "경찰서", "지구대", "파출소",
    ],
}

# ★ 수집할 스트레스 분류 선택
SELECTED_STRESS = ["소음"]  # 전체 수집
# SELECTED_STRESS = ["환경"]  # 특정 하나만 수집

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SEARCH_API_URL = "https://openapi.naver.com/v1/search/local.json"


OUTPUT_DIR = "stress_solutions"

FORCE_RECRAWL = True  # True면 progress를 무시하고 강제 재수집
PROGRESS_FILE = "naver_stress_progress.json"


# 서울 25개 자치구
SEOUL_DISTRICTS = [
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# API 헤더
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def get_headers() -> Dict:
    return {

        "X-Naver-Client-Id":     NAVER_CLIENT_ID,

        "X-Naver-Client-Secret": NAVER_CLIENT_SECRET,

    }


def test_api_key() -> bool:
    """API 키 유효성 검증"""
    print("🔑 네이버 API 키 검증 중...")

    
    if NAVER_CLIENT_ID in ("", "여기에_본인의_Client_ID_입력"):

        print("  ❌ Client ID가 입력되지 않았습니다.")

        print("     → 코드 상단에 네이버 API 키를 입력하세요.\n")

        return False

    
    try:
        resp = requests.get(
            SEARCH_API_URL,
            headers=get_headers(),
            params={"query": "강남역", "display": 1},
            timeout=10,
        )
        if resp.status_code == 200:
            print("  ✅ API 키 정상\n")

            return True

        elif resp.status_code == 401:
            print("  ❌ 401 Unauthorized — API 키가 잘못되었습니다.\n")

        elif resp.status_code == 403:
            print("  ❌ 403 Forbidden — API 권한이 없습니다.\n")

        else:
            print(f"  ❌ HTTP {resp.status_code}\n")

    except Exception as e:

        print(f"  ❌ 오류: {e}\n")

    return False



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 진행상황 관리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_progress() -> Dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:

            return json.load(f)

    return {"completed": {}}



def save_progress(progress: Dict):
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:

        json.dump(progress, f, ensure_ascii=False, indent=2)



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 검색 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def search_places(query: str, start: int = 1, display: int = 10) -> List[Dict]:

    """
    네이버 로컬 검색 API 호출
    - display: 한 번에 최대 10건
    - start: 페이지 시작 위치 (최대 100)
    - sort: comment (리뷰 많은 순)

    """
    params = {
        "query":   query,
        "display": display,
        "start":   start,
        "sort":    "comment",  # 리뷰 많은 순 = 인기 장소

    }
    
    resp = requests.get(SEARCH_API_URL, headers=get_headers(), params=params, timeout=10)
    
    if resp.status_code != 200:
        return []

    
    data = resp.json()

    return data.get("items", [])



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 파싱 및 품질 검증
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def clean_text(text: str) -> str:

    """HTML 태그 제거"""
    return text.replace("<b>", "").replace("</b>", "")



def parse_item(item: Dict, district: str, stress_type: str, keyword: str) -> Dict:
    """검색 결과를 표준 형식으로 변환"""
    # 네이버 좌표: 카텍(KATEC) 좌표계 (정수) → WGS84로 변환 필요
    # 간단 변환: mapx / 10000000, mapy / 10000000
    mapx = item.get("mapx", "")
    mapy = item.get("mapy", "")
    
    lng = float(mapx) / 10000000 if mapx else 0

    lat = float(mapy) / 10000000 if mapy else 0
    
    return {

        "자치구":      district,
        "스트레스분류": stress_type,
        "해결태그":    f"키워드:{keyword}",
        "장소명":      clean_text(item.get("title", "")),

        "카테고리":    item.get("category", ""),
        "도로명주소":  item.get("roadAddress", ""),
        "경도":       lng,

        "위도":       lat,
        "전화번호":    item.get("telephone", ""),

        "플레이스URL": item.get("link", ""),

        "_place_key": f"{clean_text(item.get('title', ''))}_{item.get('roadAddress', '')}",  # 중복 제거용

    }


def is_valid_place(place: Dict, district: str) -> bool:
    """장소 유효성 검증"""
    # 1. 도로명주소에 자치구 포함 여부
    road_address = place.get("도로명주소", "")
    if road_address and district not in road_address:

        return False

    
    # 2. 필수 정보 존재
    if not place.get("장소명") or not place.get("경도") or not place.get("위도"):

        return False

    
    # 3. 서울시 좌표 범위 검증
    lng, lat = place["경도"], place["위도"]

    if not (126.7 <= lng <= 127.2 and 37.4 <= lat <= 37.7):

        return False

    
    return True



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 자치구 × 스트레스 분류별 수집
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def crawl_district_stress(
    district: str,
    stress_type: str,
    keywords: List[str],
) -> List[Dict]:
    """한 자치구의 특정 스트레스 해결책 수집"""
    results = []
    seen_keys = set()

    
    print(f"\n  [{stress_type}] 수집 시작")

    
    for keyword in keywords:

        query = f"서울 {district} {keyword}"
        fetched = 0
        
        # 네이버 API: 최대 start=100까지, display=10씩
        # 1페이지 = 최대 10건 수집 (효율 고려)
        for start in [1]:  # 1, 6 (2페이지)

            try:
                items = search_places(query, start=start, display=10)
            except Exception as e:

                print(f"    [오류] {keyword}: {e}")

                break
            
            if not items:

                break
            
            for item in items:

                place = parse_item(item, district, stress_type, keyword)
                
                # 중복 체크 (장소명 + 도로명주소 조합)
                key = place["_place_key"]
                if key in seen_keys:

                    continue

                
                # 유효성 검증
                if not is_valid_place(place, district):

                    continue

                
                seen_keys.add(key)

                results.append(place)

                fetched += 1
            
            time.sleep(0.3)  # API 초당 10건 제한 준수
        
        print(f"    키워드[{keyword}]: {fetched}건")

        time.sleep(0.5)
    
    print(f"  [{stress_type}] 총 {len(results)}건 수집")

    return results



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전체 수집
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def crawl_all_solutions():
    summary_df = pd.DataFrame()
    """서울 25개 자치구 × 선택한 스트레스 분류별 해결책 수집"""

    if not test_api_key():

        print("❌ 크롤링을 중단합니다. API 키를 확인하세요.")

        return None

    
    # 출력 폴더 생성
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 진행상황 로드
    progress = load_progress()
    if FORCE_RECRAWL:
        print("[INFO] 강제 재수집 모드: 기존 진행상황을 무시합니다.")
    

    # 수집 시작
    all_results = {stress: [] for stress in SELECTED_STRESS}

    
    for i, district in enumerate(SEOUL_DISTRICTS, 1):

        print(f"\n{'='*60}")

        print(f"[{i:02d}/25] 🏘️  {district}")

        print(f"{'='*60}")

        
        for stress_type in SELECTED_STRESS:

            # 이미 수집 완료한 경우 건너뛰기
            key = f"{district}_{stress_type}"
            if (not FORCE_RECRAWL) and progress["completed"].get(key):

                print(f"  [{stress_type}] ⏭️  이미 수집 완료 (건너뜀)")

                continue

            
            # 해당 스트레스 키워드 가져오기
            keywords = STRESS_TAGS[stress_type]
            
            # 수집
            results = crawl_district_stress(district, stress_type, keywords)
            all_results[stress_type].extend(results)

            
            # 진행상황 저장
            progress["completed"][key] = True
            save_progress(progress)
            
            time.sleep(1)
    
    # 스트레스 분류별 CSV 저장
    print(f"\n{'='*60}")

    print("📊 결과 저장 중...")

    print(f"{'='*60}")

    
    summary = []
    
    for stress_type in SELECTED_STRESS:

        if not all_results[stress_type]:

            print(f"  ⚠️  [{stress_type}] 수집된 데이터 없음")

            continue

        
        df = pd.DataFrame(all_results[stress_type])
        
        # 중복 제거
        df.drop_duplicates(subset=["_place_key"], inplace=True)

        
        # 최종 출력 컬럼만 선택
        output_columns = [

            "자치구", "스트레스분류", "해결태그", "장소명", "카테고리",
            "도로명주소", "경도", "위도", "전화번호", "플레이스URL"
        ]
        df = df[output_columns]

        df.reset_index(drop=True, inplace=True)

        
        # 파일 저장
        filename = f"{stress_type}_해결책_네이버.csv"

        filepath = os.path.join(OUTPUT_DIR, filename)

        df.to_csv(filepath, index=False, encoding="utf-8-sig")

        
        print(f"  ✅ [{stress_type}] {filepath} ({len(df)}건)")

        
        summary.append({

            "스트레스분류": stress_type,
            "수집건수": len(df),

            "자치구수": df["자치구"].nunique(),

            "파일경로": filepath,
        })
    
    # 요약 정보
    if summary:
        summary_df = pd.DataFrame(summary)
        summary_path = os.path.join(OUTPUT_DIR, "_수집_요약_네이버.csv")

        summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

        
        print(f"\n{'='*60}")

        print("✅ 전체 수집 완료!")

        print(f"{'='*60}")

    if summary_df is not None and not summary_df.empty:
        print(summary_df.to_string(index=False))

        print(f"\n📁 저장 위치: {os.path.abspath(OUTPUT_DIR)}/")

    if summary_df is None:
        summary_df = pd.DataFrame()
    return summary_df

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if __name__ == "__main__":

    print(f"""

{'='*60}
서울의 기분 프로젝트 — 스트레스 해결책 크롤러 (네이버)
{'='*60}
수집 대상 스트레스: {', '.join(SELECTED_STRESS)}

수집 범위: 서울 25개 자치구
{'='*60}
""")
    
    start_time = datetime.now()

    summary = crawl_all_solutions()

    end_time = datetime.now()

    
    if summary is not None:

        print(f"\n⏱️  소요 시간: {end_time - start_time}")







서울의 기분 프로젝트 — 스트레스 해결책 크롤러 (네이버)
수집 대상 스트레스: 소음

수집 범위: 서울 25개 자치구

🔑 네이버 API 키 검증 중...
  ✅ API 키 정상

[INFO] 강제 재수집 모드: 기존 진행상황을 무시합니다.

[01/25] 🏘️  강남구

  [소음] 수집 시작
    키워드[조용한 카페]: 5건
    키워드[도서관]: 5건
    키워드[북카페]: 3건
    키워드[한적한 카페]: 0건
    키워드[미술관]: 5건
    키워드[박물관]: 5건
    키워드[전시회]: 3건
    키워드[힐링]: 5건
    키워드[수목원]: 3건
    키워드[식물원]: 5건
    키워드[도심속자연]: 0건
    키워드[조용한]: 3건
    키워드[차분한]: 0건
    키워드[혼자 가기 좋은]: 1건
    키워드[조용히]: 0건
    키워드[한적한]: 0건
    키워드[여유로운]: 0건
    키워드[산책코스]: 3건
  [소음] 총 46건 수집

[02/25] 🏘️  강동구

  [소음] 수집 시작
    키워드[조용한 카페]: 5건
    키워드[도서관]: 5건
    키워드[북카페]: 4건
    키워드[한적한 카페]: 0건
    키워드[미술관]: 3건
    키워드[박물관]: 5건
    키워드[전시회]: 5건
    키워드[힐링]: 5건
    키워드[수목원]: 3건
    키워드[식물원]: 5건
    키워드[도심속자연]: 0건
    키워드[조용한]: 4건
    키워드[차분한]: 0건
    키워드[혼자 가기 좋은]: 1건
    키워드[조용히]: 1건
    키워드[한적한]: 0건
    키워드[여유로운]: 0건
    키워드[산책코스]: 3건
  [소음] 총 49건 수집

[03/25] 🏘️  강북구

  [소음] 수집 시작
    키워드[조용한 카페]: 5건
    키워드[도서관]: 5건
    키워드[북카페]: 5건
    키워드[한적한 카페]: 0건
    키워드[미술관]: 4건
    키워

In [18]:
# === 품질 필터: 태그-카테고리 불일치/불필요 장소 제외 (Naver) ===

import re
import pandas as pd


# 1) 전역 제외어: 장소명/카테고리/주소에 아래 단어가 들어가면 제외
GLOBAL_EXCLUDE_PATTERNS = [

    r"주차장", r"화장실", r"개방화장실", r"교차로",
    r"충전소", r"전기차", r"주유소", r"정류장", r"톨게이트",
    r"세차장", r"정비소", r"스터디카페", r"독서실"
]

# 2) 태그별 허용 카테고리(네이버 카테고리 포맷 기준)
TAG_ALLOW_PATTERNS = {

    "키워드:도서관": [r"도서관"],
    "키워드:미술관": [r"미술관", r"전시관", r"갤러리", r"화랑", r"문화시설"],
    "키워드:박물관": [r"박물관", r"유적지", r"사적지", r"기념관"],
    "키워드:북카페": [r"북카페", r"서점", r"도서관", r"카페"],
    "키워드:감성카페": [r"카페", r"베이커리", r"디저트", r"브런치"],
    "키워드:힐링카페": [r"카페", r"베이커리", r"디저트", r"브런치"],
    "키워드:조용한": [r"카페", r"도서관"],
    "키워드:조용한 카페": [r"카페", r"베이커리", r"디저트"],
    "키워드:집중": [r"도서관", r"카페", r"북카페"]
}

# 3) 태그별 제외 카테고리(허용어가 있어도 우선 제외)
TAG_BLOCK_PATTERNS = {

    "키워드:미술관": [r"한식", r"고기요리", r"주유소", r"주차장", r"충전소"],
    "키워드:박물관": [r"한식", r"고기요리", r"주유소", r"주차장", r"충전소"],
    "키워드:북카페": [r"학원", r"영어학원", r"수학학원"],
    "키워드:집중": [r"대출", r"캐피탈", r"충전소", r"주유소"]
}


def _norm_text(*vals):

    txt = " ".join([str(v) for v in vals if pd.notna(v)]).strip().lower()

    return re.sub(r"\s+", " ", txt)



def is_globally_excluded(row):
    txt = _norm_text(row.get("장소명"), row.get("카테고리"), row.get("도로명주소"))

    return any(re.search(p, txt) for p in GLOBAL_EXCLUDE_PATTERNS)



def is_tag_category_matched(row):
    tag = str(row.get("해결태그", ""))
    cat = str(row.get("카테고리", ""))

    # blocklist 우선 제외
    blocks = TAG_BLOCK_PATTERNS.get(tag, [])

    if any(re.search(p, cat) for p in blocks):

        return False


    # allowlist 검사
    allow = TAG_ALLOW_PATTERNS.get(tag)

    if not allow:

        # 정의되지 않은 태그는 기본 통과
        return True

    return any(re.search(p, cat) for p in allow)



def filter_places(df):
    df = df.copy()

    # 결측 카테고리는 매칭 불가로 처리
    if "카테고리" in df.columns:

        df["카테고리"] = df["카테고리"].fillna("")


    df["_global_excluded"] = df.apply(is_globally_excluded, axis=1)
    df["_tag_match"] = df.apply(is_tag_category_matched, axis=1)

    clean_df = df[(~df["_global_excluded"]) & (df["_tag_match"])].copy()

    removed_df = df[(df["_global_excluded"]) | (~df["_tag_match"])].copy()

    # 중복 제거(같은 구/태그/장소/좌표)
    dedup_cols = ["자치구", "해결태그", "장소명", "경도", "위도"]
    dedup_cols = [c for c in dedup_cols if c in clean_df.columns]

    if dedup_cols:
        clean_df = clean_df.drop_duplicates(subset=dedup_cols)


    return clean_df, removed_df



# 사용 예시
raw = pd.read_csv("stress_solutions/소음_해결책_네이버.csv")

clean, removed = filter_places(raw)

clean.to_csv("stress_solutions/소음_해결책_네이버_정제.csv", index=False, encoding="utf-8-sig")

removed.to_csv("stress_solutions/소음_해결책_네이버_제외목록.csv", index=False, encoding="utf-8-sig")

print(f"원본: {len(raw)} / 정제: {len(clean)} / 제외: {len(removed)}")

print("제외 사유 요약:")

print(removed[["해결태그", "장소명", "카테고리"]].head(20).to_string(index=False))




원본: 648 / 정제: 605 / 제외: 43
제외 사유 요약:
      해결태그                    장소명        카테고리
   키워드:박물관                 스페이스 씨   문화,예술>전시관
   키워드:박물관                크리에이티브통   문화,예술>전시관
   키워드:조용한       야나기일식집 초밥스시 청담역점  음식점>일식>일식당
   키워드:조용한              어거스트힐 강남점      음식점>양식
   키워드:박물관       서울 암사동 유적 박물관전시실   문화,예술>전시관
   키워드:조용한              모로미 강서구청점     음식점>일식당
키워드:조용한 카페                    사생활 음식점>패밀리레스토랑
   키워드:조용한            동경산책 서울대입구점  음식점>일식>일식당
   키워드:조용한                   서울초밥 음식점>일식>초밥,롤
   키워드:조용한                   탄토탄토  음식점>이탈리아음식
   키워드:박물관    평화박물관건립추진위원회_스페이스99   문화,예술>전시관
   키워드:조용한        이자카야나무 구로디지털단지점 음식점>술집>이자카야
   키워드:박물관 금천 순이의 집 구로공단 노동자생활체험관   문화,예술>전시관
   키워드:조용한                미분당 가산점   음식점>베트남음식
   키워드:조용한                경복궁 창동점 음식점>육류,고기요리
   키워드:조용한                미분당 창동점   음식점>베트남음식
   키워드:박물관              한국경제발전전시관   문화,예술>전시관
   키워드:조용한               미분당 노량진점   음식점>베트남음식
   키워드:조용한            미분당 신대방삼거리점   음식점>베트남음식
   키워드:조용한                연남토마 본점    음식점>퓨전

In [19]:
# === 소음 확장 태그용 필수/제외 규칙 (override v3) ===
# 변경사항: 스터디카페/독서실 전면 제외

import re
import pandas as pd

GLOBAL_EXCLUDE_PATTERNS = [
    r"주차장", r"화장실", r"개방화장실", r"교차로", r"로터리",
    r"충전소", r"전기차", r"주유소", r"정비소", r"세차장", r"톨게이트",
    r"빌라", r"오피스텔", r"아파트", r"주택",
    r"약국", r"안과", r"내과", r"치과", r"한의원", r"의원", r"병원",
    r"대출", r"캐피탈", r"보험", r"은행",
    r"행사대행", r"전시,행사대행", r"휴대폰액세서리", r"기능성화장품",
    r"애견호텔", r"강아지유치원", r"반려동물호텔",
    r"스터디카페", r"독서실"
]

TAG_ALLOW_PATTERNS = {
    "키워드:조용한 카페": [r"카페", r"디저트", r"베이커리", r"브런치", r"음식점"],
    "키워드:한적한 카페": [r"카페", r"디저트", r"베이커리", r"브런치", r"음식점"],
    "키워드:도서관": [r"도서관", r"작은도서관", r"군구립도서관"],
    "키워드:북카페": [r"북카페", r"서점", r"카페", r"도서관"],
    "키워드:미술관": [r"미술관", r"갤러리", r"화랑", r"전시관", r"문화시설"],
    "키워드:박물관": [r"박물관", r"전시관", r"기념관", r"유적지", r"사적지", r"문화시설"],
    "키워드:전시회": [r"전시", r"갤러리", r"미술관", r"전시관", r"박물관", r"문화시설"],
    "키워드:힐링": [r"공원", r"수목원", r"식물원", r"산책", r"둘레길", r"명소", r"카페", r"음식점"],
    "키워드:수목원": [r"수목원", r"식물원", r"공원", r"명소", r"생태"],
    "키워드:식물원": [r"식물원", r"수목원", r"공원", r"명소", r"생태"],
    "키워드:도심속자연": [r"공원", r"생태", r"산책", r"둘레길", r"명소", r"숲"],
    "키워드:산책코스": [r"산책", r"공원", r"둘레길", r"명소", r"하천", r"수변"],
    "키워드:조용한": [r"카페", r"도서관", r"북카페", r"공원", r"음식점"],
    "키워드:차분한": [r"카페", r"도서관", r"북카페", r"미술관", r"전시", r"공원", r"음식점"],
    "키워드:혼자 가기 좋은": [r"카페", r"도서관", r"북카페", r"미술관", r"공원", r"산책", r"음식점"],
    "키워드:조용히": [r"카페", r"도서관", r"북카페", r"공원", r"음식점"],
    "키워드:한적한": [r"카페", r"공원", r"산책", r"수목원", r"식물원", r"명소", r"음식점"],
    "키워드:여유로운": [r"카페", r"공원", r"산책", r"미술관", r"전시", r"명소", r"음식점"]
}

TAG_BLOCK_PATTERNS = {
    "키워드:미술관": [r"한식", r"중식", r"일식", r"고기요리", r"주유소", r"주차장", r"충전소", r"학원", r"스터디카페", r"독서실"],
    "키워드:박물관": [r"한식", r"중식", r"일식", r"고기요리", r"주유소", r"주차장", r"충전소", r"화장실", r"스터디카페", r"독서실"],
    "키워드:전시회": [r"행사대행", r"지원,대행", r"한식", r"중식", r"일식", r"고기요리", r"주유소", r"주차장", r"충전소", r"스터디카페", r"독서실"],
    "키워드:힐링": [r"병원", r"의원", r"안과", r"내과", r"치과", r"한의원", r"휴대폰액세서리", r"기능성화장품", r"대출", r"캐피탈", r"주유소", r"충전소", r"교차로", r"스터디카페", r"독서실"],
    "키워드:수목원": [r"한식", r"중식", r"일식", r"닭요리", r"주택", r"빌라", r"주유소", r"주차장", r"충전소", r"꽃집", r"스터디카페", r"독서실"],
    "키워드:식물원": [r"꽃집", r"꽃배달", r"주택", r"빌라", r"주유소", r"주차장", r"충전소", r"스터디카페", r"독서실"],
    "키워드:조용한 카페": [r"클럽", r"술집", r"호프", r"포차", r"스터디카페", r"독서실"],
    "키워드:한적한 카페": [r"클럽", r"술집", r"호프", r"포차", r"스터디카페", r"독서실"],
    "키워드:도서관": [r"스터디카페", r"독서실"],
    "키워드:북카페": [r"스터디카페", r"독서실"],
    "키워드:조용한": [r"대출", r"캐피탈", r"주유소", r"충전소", r"스터디카페", r"독서실"],
    "키워드:차분한": [r"대출", r"캐피탈", r"주유소", r"충전소", r"스터디카페", r"독서실"],
    "키워드:여유로운": [r"대출", r"캐피탈", r"주유소", r"충전소", r"스터디카페", r"독서실"]
}


def _norm_text(*vals):
    txt = " ".join([str(v) for v in vals if pd.notna(v)]).strip().lower()
    return re.sub(r"\s+", " ", txt)


def is_globally_excluded(row):
    txt = _norm_text(row.get("장소명"), row.get("카테고리"), row.get("도로명주소"))
    return any(re.search(p, txt) for p in GLOBAL_EXCLUDE_PATTERNS)


def is_tag_category_matched(row):
    tag = str(row.get("해결태그", ""))
    cat = str(row.get("카테고리", ""))

    blocks = TAG_BLOCK_PATTERNS.get(tag, [])
    if any(re.search(p, cat) for p in blocks):
        return False

    allow = TAG_ALLOW_PATTERNS.get(tag)
    if not allow:
        return True
    return any(re.search(p, cat) for p in allow)


def filter_places(df):
    df = df.copy()
    if "카테고리" in df.columns:
        df["카테고리"] = df["카테고리"].fillna("")

    df["_global_excluded"] = df.apply(is_globally_excluded, axis=1)
    df["_tag_match"] = df.apply(is_tag_category_matched, axis=1)

    clean_df = df[(~df["_global_excluded"]) & (df["_tag_match"])].copy()
    removed_df = df[(df["_global_excluded"]) | (~df["_tag_match"])].copy()

    dedup_cols = ["자치구", "해결태그", "장소명", "경도", "위도"]
    dedup_cols = [c for c in dedup_cols if c in clean_df.columns]
    if dedup_cols:
        clean_df = clean_df.drop_duplicates(subset=dedup_cols)

    return clean_df, removed_df

print("[OK] 소음 확장 태그 규칙 v3 적용 완료 (스터디카페/독서실 전면 제외)")


[OK] 소음 확장 태그 규칙 v3 적용 완료 (스터디카페/독서실 전면 제외)
